In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/flight_information/flights_info_gold.parquet")

In [3]:
df_traj = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet")

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
df.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul', 'wake_turbulence_cat'],
      dtype='str')

In [6]:
df_traj.columns

Index(['flight_id', 'episode_id', 'point_timestamp', 'icao', 'date', 'e_m',
       'n_m', 'u_m', 'ux', 'uy', 'uz', 'r', 'sin_theta', 'cos_theta',
       'delta_t', 'distance_km', 'on_ground', 'segment_id', 'gap_flag'],
      dtype='str')

In [7]:
len(df)

83727

In [8]:
df_flights = df.copy()
df_traj = df_traj.copy()

In [10]:
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────
FLIGHTS_PATH      = "/Users/YGT/ist-airport-decision-support-system/data/gold/flight_information/flights_info_gold.parquet"
TRAJ_PATH         = "/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet"

# ─────────────────────────────────────────────────────────────
# PARAMETRELER
# ─────────────────────────────────────────────────────────────
LANDING_WINDOW_HRS    = 4      # ref_time ± saat
ENTRY_LOOKBACK_HRS    = 3      # landing'den geriye bakış
GAP_THRESHOLD_S       = 1800   # 30 dk — yeni segment başlangıcı

POST_TERM_MIN_MIN     = 0      # plausible target aralığı (dk)
POST_TERM_MAX_MIN     = 120
ENTRY_DIST_MIN_KM     = 10     # TMA giriş mesafesi aralığı (km)
ENTRY_DIST_MAX_KM     = 125
LANDING_DIST_MAX_KM   = 20     # landing proxy max terminal mesafesi (km)
TIME_DIFF_MAX_MIN     = 60     # |landing_t − ref_time| max (dk)

# ─────────────────────────────────────────────────────────────
# 1. VERİ YÜKLE
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("1. VERİ YÜKLEME")
print("=" * 60)

df_flights = pd.read_parquet(FLIGHTS_PATH)
df_traj    = pd.read_parquet(TRAJ_PATH)
print(f"  Flights : {len(df_flights):>10,}")
print(f"  Traj    : {len(df_traj):>10,}")

# ─────────────────────────────────────────────────────────────
# 2. TİP / FORMAT
# ─────────────────────────────────────────────────────────────
df_flights["arr_sched_time_utc"]   = pd.to_datetime(df_flights["arr_sched_time_utc"],   utc=True, errors="coerce")
df_flights["arr_revised_time_utc"] = pd.to_datetime(df_flights["arr_revised_time_utc"], utc=True, errors="coerce")
df_traj["point_timestamp"]         = pd.to_datetime(df_traj["point_timestamp"],         utc=True, errors="coerce")

df_flights["hex_icao"] = df_flights["hex_icao"].astype(str).str.upper().str.strip()
df_traj["icao"]        = df_traj["icao"].astype(str).str.upper().str.strip()
df_traj["distance_km"] = pd.to_numeric(df_traj["distance_km"], errors="coerce")

# ref_time: revised varsa onu kullan, yoksa sched
df_flights["ref_time"] = df_flights["arr_revised_time_utc"].fillna(
    df_flights["arr_sched_time_utc"]
)



# Eksik değerleri düşür
df_flights = df_flights.dropna(subset=["hex_icao", "arr_sched_time_utc", "ref_time"]).copy()
df_traj    = df_traj.dropna(subset=["icao", "point_timestamp", "distance_km"]).copy()

print(f"\n  Flights (temiz) : {len(df_flights):>10,}")
print(f"  Traj    (temiz) : {len(df_traj):>10,}")

# ─────────────────────────────────────────────────────────────
# 3. TRAJ GRUPLAMA  (ICAO bazında dict — hız için)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. TRAJ GRUPLAMA")
print("=" * 60)

df_traj = df_traj.sort_values(["icao", "point_timestamp"], kind="mergesort")
traj_grouped = {
    icao: grp.reset_index(drop=True)
    for icao, grp in df_traj.groupby("icao", sort=False)
}
print(f"  Benzersiz ICAO : {len(traj_grouped):,}")

# ─────────────────────────────────────────────────────────────
# 4. LABEL EXTRACTION
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. LABEL EXTRACTION")
print("=" * 60)

_NAT = pd.NaT
_NAN = np.nan

results = []

for i, flight in enumerate(df_flights.itertuples(index=False), start=1):
    if i % 10_000 == 0:
        print(f"  [{i:,} / {len(df_flights):,}]")

    icao     = flight.hex_icao
    sched    = flight.arr_sched_time_utc
    ref_time = flight.ref_time

    row = dict(
        actual_landing_time         = _NAT,
        landing_dist_km             = _NAN,
        actual_entry_time           = _NAT,
        entry_dist_km               = _NAN,
        post_terminal_duration_s    = _NAN,
        post_terminal_duration_min  = _NAN,
        landing_proxy_time_diff_min = _NAN,
        is_matched                  = False,
        is_plausible_label          = False,
        match_quality_flag          = "no_match",
    )

    if icao not in traj_grouped:
        results.append(row)
        continue

    grp = traj_grouped[icao]

    # ── A) LANDING PROXY ──────────────────────────────────────
    t_lo = ref_time - pd.Timedelta(hours=LANDING_WINDOW_HRS)
    t_hi = ref_time + pd.Timedelta(hours=LANDING_WINDOW_HRS)

    win_land = grp[
        (grp["point_timestamp"] >= t_lo) &
        (grp["point_timestamp"] <= t_hi)
    ]

    if len(win_land) == 0:
        results.append(row)
        continue

    best      = win_land.loc[win_land["distance_km"].idxmin()]
    landing_t = best["point_timestamp"]
    landing_d = float(best["distance_km"])
    time_diff = (landing_t - ref_time).total_seconds() / 60.0

    row["actual_landing_time"]         = landing_t
    row["landing_dist_km"]             = landing_d
    row["landing_proxy_time_diff_min"] = time_diff
    row["is_matched"]                  = True

    # ── B) ENTRY PROXY ───────────────────────────────────────
    e_lo = landing_t - pd.Timedelta(hours=ENTRY_LOOKBACK_HRS)

    win_entry = (
        grp[
            (grp["point_timestamp"] >= e_lo) &
            (grp["point_timestamp"] <= landing_t)
        ]
        .sort_values("point_timestamp")
        .reset_index(drop=True)
    )

    if len(win_entry) < 2:
        results.append(row)
        continue

    win_entry["_gap_s"] = win_entry["point_timestamp"].diff().dt.total_seconds()
    gap_pos = np.where(win_entry["_gap_s"].fillna(0).values > GAP_THRESHOLD_S)[0]

    segment = win_entry.iloc[gap_pos[-1]:] if len(gap_pos) > 0 else win_entry

    if len(segment) == 0:
        results.append(row)
        continue

    first   = segment.iloc[0]
    entry_t = first["point_timestamp"]
    entry_d = float(first["distance_km"])

    row["actual_entry_time"] = entry_t
    row["entry_dist_km"]     = entry_d

    # ── C) TARGET ─────────────────────────────────────────────
    # y_i = T_arr_actual − T_entry_actual  (makale Eq. 13)
    post_s   = (landing_t - entry_t).total_seconds()
    post_min = post_s / 60.0

    row["post_terminal_duration_s"]   = post_s
    row["post_terminal_duration_min"] = post_min

    # ── D) PLAUSIBILITY ───────────────────────────────────────
    ok = (
        POST_TERM_MIN_MIN <= post_min  <= POST_TERM_MAX_MIN and
        ENTRY_DIST_MIN_KM <= entry_d   <= ENTRY_DIST_MAX_KM and
        landing_d         <= LANDING_DIST_MAX_KM            and
        abs(time_diff)    <= TIME_DIFF_MAX_MIN
    )

    row["is_plausible_label"] = ok

    if ok:
        row["match_quality_flag"] = "good_match"
    elif landing_d > LANDING_DIST_MAX_KM:
        row["match_quality_flag"] = "bad_landing_distance"
    elif abs(time_diff) > TIME_DIFF_MAX_MIN:
        row["match_quality_flag"] = "bad_time_alignment"
    elif not (ENTRY_DIST_MIN_KM <= entry_d <= ENTRY_DIST_MAX_KM):
        row["match_quality_flag"] = "bad_entry_distance"
    elif not (POST_TERM_MIN_MIN <= post_min <= POST_TERM_MAX_MIN):
        row["match_quality_flag"] = "bad_post_terminal_duration"
    else:
        row["match_quality_flag"] = "other"

    results.append(row)

# ─────────────────────────────────────────────────────────────
# 5. BİRLEŞTİR
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. BİRLEŞTİR & KAYDET")
print("=" * 60)

df_res   = pd.DataFrame(results)
df_final = pd.concat(
    [df_flights.drop(columns=["ref_time"]).reset_index(drop=True),
     df_res.reset_index(drop=True)],
    axis=1,
)
df_labeled = df_final[df_final["is_plausible_label"]].copy()


# ─────────────────────────────────────────────────────────────
# 6. RAPOR
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("5. RAPOR")
print("=" * 60)

total   = len(df_final)
matched = df_final["is_matched"].sum()
labeled = len(df_labeled)

print(f"  Toplam flight       : {total:>10,}")
print(f"  Landing bulunan     : {df_final['actual_landing_time'].notna().sum():>10,}  ({df_final['actual_landing_time'].notna().sum()/total*100:.1f}%)")
print(f"  Entry bulunan       : {df_final['actual_entry_time'].notna().sum():>10,}  ({df_final['actual_entry_time'].notna().sum()/total*100:.1f}%)")
print(f"  Plausible label     : {labeled:>10,}  ({labeled/total*100:.1f}%)")

print("\n  Match quality:")
print(df_final["match_quality_flag"].value_counts(dropna=False).to_string())

pct = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

print("\n  TARGET — post_terminal_duration_min:")
print(df_labeled["post_terminal_duration_min"].describe(percentiles=pct).round(2).to_string())

print("\n  entry_dist_km:")
print(df_labeled["entry_dist_km"].describe(percentiles=pct).round(2).to_string())

print("\n  landing_dist_km:")
print(df_labeled["landing_dist_km"].describe(percentiles=pct).round(2).to_string())

1. VERİ YÜKLEME
  Flights :     83,727
  Traj    : 28,425,192

  Flights (temiz) :     83,727
  Traj    (temiz) : 28,425,192

2. TRAJ GRUPLAMA
  Benzersiz ICAO : 10,009

3. LABEL EXTRACTION
  [10,000 / 83,727]
  [20,000 / 83,727]
  [30,000 / 83,727]
  [40,000 / 83,727]
  [50,000 / 83,727]
  [60,000 / 83,727]
  [70,000 / 83,727]
  [80,000 / 83,727]

4. BİRLEŞTİR & KAYDET

5. RAPOR
  Toplam flight       :     83,727
  Landing bulunan     :     66,016  (78.8%)
  Entry bulunan       :     56,377  (67.3%)
  Plausible label     :     35,891  (42.9%)

  Match quality:
match_quality_flag
good_match              35891
no_match                27350
bad_landing_distance    12190
bad_time_alignment       8294
bad_entry_distance          2

  TARGET — post_terminal_duration_min:
count    35891.00
mean        11.35
std          7.32
min          0.00
1%           1.13
5%           1.58
25%          5.57
50%         10.60
75%         16.22
95%         24.73
99%         30.13
max         56.73

  entr

In [11]:
len(df_labeled)

35891

In [12]:
len(df_final)

83727

In [13]:
df_labeled.head()

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,dep_lat,dep_lon,dep_altitude,distance,haul,wake_turbulence_cat,actual_landing_time,landing_dist_km,actual_entry_time,entry_dist_km,post_terminal_duration_s,post_terminal_duration_min,landing_proxy_time_diff_min,is_matched,is_plausible_label,match_quality_flag
0,2025-07-31,Thursday,21,728678,Airbus A320,YI-ASX,I A W,IA 223,IAW223,IA,IAW,BGW,ORBI,Baghdad,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:00:00+00:00,2025-07-31 18:00:00+00:00,Arrived,33.262501,44.234600,114.0,1630.418823,medium-haul,M,2025-07-31 17:43:43+00:00,10.675306,2025-07-31 17:29:23+00:00,63.968685,860.0,14.333333,-16.283333,True,True,good_match
1,2025-07-31,Thursday,21,7114EF,Airbus A321,UNKNOWN,Saudi Arabian,SV 261,SVA261,SV,SVA,JED,OEJN,Jeddah,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:10:00+00:00,Arrived,21.680241,39.157436,48.0,2387.484375,medium-haul,M,2025-07-31 17:51:36+00:00,7.054359,2025-07-31 17:41:16+00:00,47.864044,620.0,10.333333,-18.400000,True,True,good_match
2,2025-07-31,Thursday,21,040184,Boeing 737,ET-AXG,Ethiopian,ET 722,ETH722,ET,ETH,ADD,HAAB,Addis Ababa,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:20:00+00:00,Arrived,8.977890,38.799301,7630.0,3724.830566,medium-haul,M,2025-07-31 18:23:17+00:00,12.865262,2025-07-31 18:14:19+00:00,44.443016,538.0,8.966667,3.283333,True,True,good_match
4,2025-07-31,Thursday,21,4BA91A,Airbus A321,TC-JHZ,Turkish,TK 1098,THY9GS,TK,THY,TIV,LYTV,Tivat,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:20:00+00:00,2025-07-31 18:20:00+00:00,Arrived,42.404701,18.723301,20.0,839.713257,short-haul,M,2025-07-31 18:16:03+00:00,12.850230,2025-07-31 18:11:34+00:00,34.330212,269.0,4.483333,-3.950000,True,True,good_match
5,2025-07-31,Thursday,21,4691C2,Airbus A320,SX-DNB,Aegean,A3 434,AEE434,A3,AEE,HER,LGIR,Heraklion,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:30:00+00:00,2025-07-31 18:40:00+00:00,Arrived,35.339699,25.180300,115.0,729.683655,short-haul,M,2025-07-31 18:25:19+00:00,11.428142,2025-07-31 18:21:48+00:00,30.468594,211.0,3.516667,-14.683333,True,True,good_match


In [14]:
drop_cols = [
    "arr_revised_time_utc",
    "is_matched",
    "is_plausible_label",
    "match_quality_flag",
    "delay_minutes",
]


drop_cols = [c for c in drop_cols if c in df_labeled.columns]

df_model = df_labeled.drop(columns=drop_cols).copy()

In [15]:
df_model.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'status', 'dep_lat', 'dep_lon', 'dep_altitude',
       'distance', 'haul', 'wake_turbulence_cat', 'actual_landing_time',
       'landing_dist_km', 'actual_entry_time', 'entry_dist_km',
       'post_terminal_duration_s', 'post_terminal_duration_min',
       'landing_proxy_time_diff_min'],
      dtype='str')